In [3]:
from fractions import Fraction
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFTGate

def calcular_periodo(a, N):
    T = 1
    while pow(a, T, N) != 1:
        T += 1
    return T

def c_amod15(a, power):
    if a not in [2, 7, 8, 11, 13]:
        raise ValueError()

    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(0, 1)
            U.swap(1, 2)
            U.swap(2, 3)
        if a in [7, 8]:
            U.swap(2, 3)
            U.swap(1, 2)
            U.swap(0, 1)
        if a in [4, 11]:
            U.swap(1, 3)
            U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)

    return U.to_gate(label=f"{a}^{power} mod 15").control()

def build_shor_circuit(n_count, n_aux, a):
    qc = QuantumCircuit(n_count + n_aux, n_count)

    for q in range(n_count):
        qc.h(q)
    qc.x(n_count)

    for q in range(n_count):
        qc.append(c_amod15(a, 2**q), [q] + [i + n_count for i in range(n_aux)])

    qc.append(QFTGate(n_count).inverse(), range(n_count))
    qc.measure(range(n_count), range(n_count))

    return qc

def check_k(k, Q, T_esperado, a, N):
    if k == 0:
        return False

    T_candidato = Fraction(k, Q).limit_denominator(N).denominator

    if T_candidato != T_esperado:
        return False

    if T_candidato % 2 != 0 or pow(a, T_candidato // 2, N) == (N - 1):
        return False

    return True

def main():
    N_COUNT = 4
    N_AUX = 4
    N_OBJETIVO = 15
    BASE_COPRIMA = 7
    SHOTS = 1000

    T_calculado = calcular_periodo(BASE_COPRIMA, N_OBJETIVO)
    print(f"Periodo (T) calculado analíticamente: {T_calculado}")

    qc = build_shor_circuit(N_COUNT, N_AUX, BASE_COPRIMA)

    simulador = AerSimulator()
    circuito_transpilado = transpile(qc, simulador)
    resultados = simulador.run(circuito_transpilado, shots=SHOTS).result().get_counts()

    print("\n--- RESULTADOS IDEALES (SIN RUIDO) ---")
    print("Distribución de medidas obtenidas:", resultados)

    print("\n--- EVALUACIÓN ANALÍTICA (Fracciones Continuas) ---")
    Q_total = 2 ** N_COUNT
    exitos_analiticos = 0

    for bitstring, count in resultados.items():
        k = int(bitstring, 2)
        es_exito = check_k(k, Q_total, T_calculado, BASE_COPRIMA, N_OBJETIVO)
        print(f"Medida k={k:2} (Apariciones: {count:3}) -> Check: {es_exito}")

        if es_exito:
            exitos_analiticos += count

    probabilidad_se = exitos_analiticos / SHOTS
    print(f"\nProbabilidad de éxito analítica (P_se): {probabilidad_se * 100:.2f}%")

if __name__ == "__main__":
    main()

Periodo (T) calculado analíticamente: 4

--- RESULTADOS IDEALES (SIN RUIDO) ---
Distribución de medidas obtenidas: {'1100': 236, '0100': 267, '0000': 245, '1000': 252}

--- EVALUACIÓN ANALÍTICA (Fracciones Continuas) ---
Medida k=12 (Apariciones: 236) -> Check: True
Medida k= 4 (Apariciones: 267) -> Check: True
Medida k= 0 (Apariciones: 245) -> Check: False
Medida k= 8 (Apariciones: 252) -> Check: False

Probabilidad de éxito analítica (P_se): 50.30%
